# Dag 8+9: Azure AI Foundry + Prompt Flow

**Eksamensrelevans:** Optimize Language Models for AI Applications — 25-30% af DP-100 eksamen

**Vigtig note:** Disse emner er nyere tilføjelser til DP-100 og optræder primært som **konceptuelle/vidensbaserede** spørgsmål — ikke dybe hands-on kodeopgaver. Fokus i denne notebook er forståelse og genkendelse af begreber.

**Nøglebegreber:**
- Azure AI Foundry: **Hub** vs. **Project** og deres relation
- Forholdet mellem **AI Foundry** og **Azure ML workspace**
- **Model Catalog**: hvad det er, og hvad du kan gøre med det
- **Prompt Flow**: flow-typer, connections, tools og deployment
- Integration mellem Prompt Flow og Azure ML

**Læringsmål:**
- Forklare hvad en Hub og et Project er i AI Foundry, og hvornår du bruger hvad
- Beskrive Model Catalog og de mulige deployment-scenarier
- Skelne mellem de tre Prompt Flow-typer og hvornår hver bruges
- Redegøre for connections og tools i Prompt Flow
- Forklare, hvordan Prompt Flow deployments fungerer

**Forudsætninger:** Dag 1-7 gennemført. Grundlæggende forståelse for Azure ML workspaces og deployments.

---
## DAG 8: Azure AI Foundry
---

## 1. Hvad er Azure AI Foundry?

**Azure AI Foundry** (tidligere kaldet Azure AI Studio) er Microsofts samlede platform til at bygge, træne og deploye AI-løsninger — herunder generative AI-applikationer og language models.

Foundry bygger oven på Azure ML og udvider det med:
- Adgang til **Model Catalog** (foundation models fra Microsoft, Meta, Mistral, m.fl.)
- **Prompt Flow** (visuel orkestreringsværktøj til LLM-applikationer)
- **Content Safety** og **Responsible AI** funktioner
- Unified UI via `ai.azure.com`

> **Eksamenstip:** Azure AI Foundry *er ikke* en erstatning for Azure ML — det er en udvidelse. En AI Foundry **Hub** opretter automatisk en tilknyttet Azure ML workspace i baggrunden. Du kan bruge Azure ML SDK v2 til at interagere med ressourcer, der er oprettet i Foundry.

## 2. Hub vs. Project — den vigtigste distinktion

AI Foundry har en **to-lags hierarki**: Hub og Project.

### Hub
En **Hub** er det øverste administrationsniveau:
- Ejes typisk af IT/platform-teamet
- Deler **delte ressourcer** på tværs af projekter: compute, connections (API-nøgler), netværk, storage
- Giver adgangskontrol og governance
- Svarer til et **Azure ML Workspace** under hjelmen
- Én Hub kan have **mange Projects**

### Project
Et **Project** er arbejdsrummet for et specifikt AI-projekt eller team:
- Arver delte ressourcer fra sin Hub (compute, connections, storage)
- Har sin egen **isolation**: eksperimenter, deployments, flows er per-projekt
- Data scientists og applikationsudviklere arbejder primært i Projects
- Svarer til et **child workspace** i Azure ML-terminologi

```
Hub  (IT/Platform team — delte ressourcer)
├── Project A  (Team 1 — chatbot)
├── Project B  (Team 2 — billedanalyse)
└── Project C  (Team 3 — RAG-applikation)
```

> **Eksamenstip:** Typisk eksamensscenarie: *"Hvem skal have adgang til en delt Azure OpenAI-connection på tværs af mange teams?"* — Svar: Konfigurer connectionen på **Hub**-niveau, ikke per Project. Projekter **arver** connections fra Hub.

## 3. Forholdet mellem AI Foundry og Azure ML Workspace

Dette er et klassisk eksamensemne — forstå mapningen:

| AI Foundry begreb | Azure ML begreb | Forklaring |
|---|---|---|
| **Hub** | Azure ML Workspace (kind=hub) | Øverste niveau, delte ressourcer |
| **Project** | Azure ML Workspace (kind=project) | Isoleret arbejdsrum per team |
| **Connection** | Workspace connection | Gemte credentials til external services |
| **Flow** | Pipeline (særlig type) | Prompt Flow orkestrering |
| **Deployment** | Online Endpoint | Deployede flows eller modeller |

**Vigtige fakta:**
- Når du opretter en Hub i AI Foundry, oprettes der automatisk en **Azure ML workspace** i din Azure-subscription
- Du kan bruge `MLClient` til at tilgå Project-ressourcer (modeller, jobs, endpoints)
- AI Foundry UI (`ai.azure.com`) og Azure ML Studio (`ml.azure.com`) viser overlappende, men ikke identiske, views på de samme ressourcer
- Projects kan have **egne compute-ressourcer** men kan også dele compute fra Hub

> **Eksamenstip:** Du behøver IKKE AI Foundry for at træne modeller med Azure ML. AI Foundry er primært relevant for **generative AI og LLM-workflows**. Klassisk ML (træning, hyperparameter tuning, pipelines) foregår stadig primært via Azure ML SDK.

## 4. Model Catalog

**Model Catalog** er et centralt bibliotek af pre-trained AI-modeller, som du kan browse, evaluere og deploye direkte fra AI Foundry.

### Indhold i Model Catalog
- **Microsoft-modeller**: Phi-3, Phi-4 (small language models)
- **OpenAI-modeller**: GPT-4o, GPT-4, GPT-3.5 (via Azure OpenAI)
- **Open source-modeller**: Llama 3 (Meta), Mistral, Falcon, m.fl.
- **Specialiserede modeller**: billedgenkendelse, tale, kode-generation

### Deployment-muligheder fra Model Catalog

| Metode | Beskrivelse | Hvornår? |
|---|---|---|
| **Managed Compute** | Deploy til Azure ML compute (du styrer infrastrukturen) | Open-source modeller, BYOM |
| **Serverless API** | Microsoft-hostet API, pay-per-token | Azure OpenAI og Phi-modeller |
| **Azure OpenAI Service** | Dedikeret endpoint via Azure OpenAI ressource | GPT-modeller med compliance-krav |

> **Eksamenstip:** **Serverless API** (også kaldet "Models as a Service") er nøglebegrebet her. Du deployer en model som Phi-3 eller Llama 3 og betaler per token — ingen compute-konfiguration. Kontrast til **Managed Compute** hvor du specificerer VM-størrelse og instanser (ligesom `ManagedOnlineDeployment`).

> **Eksamenstip:** Hvis eksamen spørger *"Hvilken metode bruger du for at deploye GPT-4o med de laveste driftsomkostninger og ingen infrastructure-management?"* — Svar: **Serverless API** (pay-per-token).

## 5. SDK-interaktion med AI Foundry ressourcer

Du kan bruge den velkendte `MLClient` til at interagere med AI Foundry Projects, fordi de er Azure ML workspaces under hjelmen.

**Opgave:** Opret forbindelse og list de modeller, der er tilgængelige i dit workspace/project.

*Hint:* `ml_client.models.list()` returnerer alle registrerede modeller — dette inkluderer modeller deployet fra Model Catalog.

*Hint:* For at se modeller i Model Catalog (ikke registrerede modeller) bruges AI Foundry portalen (`ai.azure.com`) — SDK'et giver ikke direkte adgang til at browse Catalog.

In [ ]:
import sys
sys.path.append("..")

from src.utils import init_ml_client
ml_client = init_ml_client()

# TODO: List alle registrerede modeller i dit workspace
# HINT: ml_client.models.list() — iterér over resultatet og print name + version

print("Registrerede modeller:")
# din kode her

## 6. Refleksion: AI Foundry koncepter

Svar på spørgsmålene herunder som kommentarer eller i en ny markdown-celle. Dette er typiske DP-100-spørgsmål.

**Spørgsmål A:** Du har tre data science-teams, der alle skal bruge den samme Azure OpenAI API-nøgle. Hvad er den korrekte måde at konfigurere dette på i AI Foundry?

**Spørgsmål B:** Hvad er den primære forskel på at deploye en model via *Serverless API* versus *Managed Compute* fra Model Catalog?

**Spørgsmål C:** En kollega siger "AI Foundry erstatter Azure ML". Er det korrekt? Forklar.

**Spørgsmål D:** Du opretter et nyt AI Foundry Hub. Hvilken Azure-ressource oprettes automatisk i din subscription som en direkte konsekvens?

**Svar:**

A. 

B. 

C. 

D. 

---
## DAG 9: Prompt Flow
---

## 7. Hvad er Prompt Flow?

**Prompt Flow** er et open-source framework (og en tjeneste i AI Foundry) til at bygge, teste og deploye **LLM-baserede applikationer**. Det giver dig en visuel og kode-baseret måde at orkestrere:
- Kald til language models (OpenAI, Llama, m.fl.)
- Python-logik til preprocessing og postprocessing
- Retrieval fra søgeindekser (RAG-patterns)
- Evalueringer og test af flow-kvalitet

Et **flow** i Prompt Flow er en DAG (Directed Acyclic Graph) af **nodes/tools**, der eksekveres i rækkefølge.

```
Input → [Python Tool] → [LLM Tool] → [Python Tool] → Output
                             ↑
                    [Prompt Template]
```

> **Eksamenstip:** Prompt Flow er tilgængeligt som:
> 1. **Tjeneste i AI Foundry** (cloud, visuel editor)
> 2. **VS Code extension** (lokal udvikling)
> 3. **`promptflow` Python-pakke** (lokal CLI og SDK)
> Eksamen spørger typisk om den cloud-baserede integration med AI Foundry og Azure ML.

## 8. Flow-typer

Der er tre flow-typer i Prompt Flow. At kende dem og hvornår de bruges er eksamensrelevant.

### Standard Flow
- **Formål:** Generel LLM-applikation med input → processing → output
- **Typiske use cases:** Tekstopsummering, klassifikation, dataekstraktion, Q&A
- **Input:** Enkelt input (f.eks. `question: str`)
- **Output:** Enkelt output (f.eks. `answer: str`)
- **Eksempel:** Bruger sender en støttehenvendelse → flow klassificerer kategori og genererer svar

### Chat Flow
- **Formål:** Multi-turn konversationer med historik
- **Specielt:** Har built-in support for **chat history** — tidligere beskeder sendes med automatisk
- **Input:** `chat_history` (liste af `{role, content}` objekter) + `question: str`
- **Output:** `answer: str`
- **Eksempel:** Kundeservice-chatbot der husker kontekst fra hele sessionen

### Evaluation Flow
- **Formål:** Evaluér kvaliteten af et andet flow's output
- **Specielt:** Tager **output fra et andet flow** + **ground truth** som input
- **Output:** Metriks (f.eks. coherence, relevance, groundedness scores 1-5)
- **Bruges til:** Batch-evaluering af Standard eller Chat flows
- **Eksempel:** Evaluér om chatbottens svar er korrekte og relevante

> **Eksamenstip:** Den vigtigste distinktion: **Chat Flow** har `chat_history` som reserveret input-navn — dette er den eneste built-in forskel fra Standard Flow strukturelt set. **Evaluation Flow** er det eneste flow-type der bruges til at *bedømme* output fra andre flows.

## 9. Connections

**Connections** i Prompt Flow gemmer credentials og konfiguration til externe services — så du ikke hardkoder API-nøgler i dine flows.

### Connection-typer

| Type | Formål | Eksempel |
|---|---|---|
| **Azure OpenAI** | Forbindelse til Azure OpenAI Service | GPT-4o API-nøgle + endpoint URL |
| **OpenAI** | Forbindelse til OpenAI.com direkte | OpenAI API-nøgle |
| **Azure AI Search** | Forbindelse til Azure AI Search | Search-endpoint + admin-nøgle |
| **Custom** | Generiske nøgle-værdi-par til egne services | Database-connection strings, egne API-nøgler |
| **Cognitive Services** | Azure AI services (Tale, Vision, m.fl.) | Cognitive Services-nøgle |

### Vigtige fakta om Connections
- Connections defineres på **Hub-niveau** og deles på tværs af Projects (eller per Project)
- Selve nøglerne gemmes i **Azure Key Vault** — Prompt Flow tilgår dem via Key Vault
- I et flow refererer du til en connection ved **navn** — ikke ved nøgle-værdier
- Connections kan markeres som **shared** (tilgængelig for alle projects under hub) eller **private**

> **Eksamenstip:** *"Hvordan sikrer du, at en Azure OpenAI API-nøgle aldrig eksponeres i din Prompt Flow kode?"* — Svar: Brug en **Azure OpenAI Connection**, som gemmer nøglen i Key Vault. Dit flow refererer til connection-navnet.

## 10. Tools i Prompt Flow

Et **tool** er en enkelt node i et flow's DAG. Hvert tool har inputs og outputs.

### De tre primære tool-typer

#### LLM Tool
- Kalder et **Language Model** (Azure OpenAI, Llama, m.fl.)
- Du specificerer: connection, deployment-navn, prompt, parametre (temperature, max_tokens)
- Input kan komme fra et **Prompt Tool** eller inline template
- Output: `string` (model-svaret)

#### Python Tool
- Kører **vilkårlig Python-kode** i en node
- Bruges til: databehandling, API-kald, parsing af LLM-output, beregninger
- Du definerer en `@tool`-dekoreret funktion med inputs og return-value
- Kan importere Python-pakker (specificeret i `requirements.txt`)

#### Prompt Tool
- Renderer en **Jinja2 prompt-template** med variable
- Output: formateret `string` til brug i et LLM Tool
- Bruges til at separere prompt-design fra flow-logik

```
# Eksempel: Python Tool i kode
from promptflow import tool

@tool
def preprocess_question(raw_input: str) -> str:
    # Rens og forbered brugerens spørgsmål
    return raw_input.strip().lower()
```

> **Eksamenstip:** *"Hvad bruges et Prompt Tool til, og hvorfor er det bedre end at skrive prompten direkte i LLM Tool?"* — Prompt Tool giver **separation of concerns**: prompten kan ændres og version-styres uafhængigt af flow-logikken. Det letter prompt-engineering og testing.

## 11. Prompt Flow og Azure ML SDK

Prompt Flow er tæt integreret med Azure ML/AI Foundry. Du kan bruge SDK'et til at:
- Oprette og administrere flows programmatisk
- Køre flows som **Azure ML jobs**
- Deploye flows til **online endpoints**

**Opgave:** Brug SDK'et til at liste eksisterende connections i dit workspace.

*Hint:* `ml_client.connections.list()` — returnerer de connections der er konfigureret i din workspace/hub.

*Hint:* Hvert connection-objekt har `.type` (f.eks. `azure_open_ai`, `cognitive_search`) og `.name`.

In [ ]:
# TODO: List alle connections i dit workspace
# HINT: ml_client.connections.list() — iterér og print name + type for hver

print("Konfigurerede connections:")
# din kode her


## 12. Deployment af Prompt Flow

Et færdigt Prompt Flow kan deployees som et **Online Endpoint** i Azure ML/AI Foundry. Det giver dig et REST API-endpoint som applikationer kan kalde.

### Deployment-processen

1. **Byg og test flow** i AI Foundry UI eller lokalt med `pf` CLI
2. **Pak flow** — Prompt Flow opretter et flow-mappe med `flow.dag.yaml`, Python-filer og `requirements.txt`
3. **Deploy til endpoint** — via AI Foundry UI ("Deploy" knap) eller via SDK
4. **Test endpointet** — kald REST API med JSON input

### Flow som Azure ML Online Endpoint

Under hjelmen deployes et Prompt Flow som en `ManagedOnlineDeployment` — det er den **samme infrastruktur** som dag 6+7. Forskellen er at Azure ML automatisk genererer scoring-scriptet fra flow-definitionen.

```
# Input-format til et deployed flow (REST API)
{
  "question": "Hvad er Azure AI Foundry?"
}

# Output-format
{
  "answer": "Azure AI Foundry er..."
}
```

> **Eksamenstip:** Et deployed Prompt Flow **er** et Online Endpoint i Azure ML — det kræver `instance_type` og `instance_count` ligesom andre deployments. Det er ikke serverless. Sammenlign med Serverless API fra Model Catalog: der betaler du per token, ikke per compute-time.

## 13. Evaluering med Evaluation Flow

**Evaluation Flows** bruges til systematisk at bedømme kvaliteten af et flows output — essentielt for MLOps med LLMs.

### Typisk evalueringsworkflow

```
Test dataset (spørgsmål + ground truth svar)
         ↓
  [Standard/Chat Flow]  ← det flow du evaluerer
         ↓
  Genererede svar
         ↓
  [Evaluation Flow]  ← bedømmer output vs. ground truth
         ↓
  Metriks: relevance=4.2, coherence=3.8, groundedness=4.5
```

### Built-in evalueringsmetriks i AI Foundry

| Metrik | Beskrivelse | Kræver ground truth? |
|---|---|---|
| **Groundedness** | Er svaret baseret på den givne kontekst? | Nej (kun kontekst) |
| **Relevance** | Er svaret relevant for spørgsmålet? | Nej |
| **Coherence** | Er svaret velformuleret og sammenhængende? | Nej |
| **Fluency** | Er sproget flydende og grammatisk korrekt? | Nej |
| **Similarity** | Ligner svaret ground truth? | Ja |
| **F1 Score** | Token-overlap med ground truth | Ja |

> **Eksamenstip:** **Groundedness** er den vigtigste metrik for RAG-applikationer — den måler om modellen "hallucinerer" eller holder sig til den retrievede kontekst. Hvis groundedness er lav, er modellen ikke pålidelig i din RAG-pipeline.

## 14. Refleksion: Prompt Flow koncepter

Svar på spørgsmålene herunder. Disse er typiske DP-100-spørgsmål om Prompt Flow.

**Spørgsmål A:** Du bygger en chatbot der skal huske brugerens tidligere beskeder. Hvilken flow-type vælger du, og hvad er det specielle ved dens input-struktur?

**Spørgsmål B:** Din Prompt Flow kalder Azure OpenAI. Hvordan sikrer du, at API-nøglen ikke er hardkodet i flow-koden?

**Spørgsmål C:** Hvad er forskellen på et **LLM Tool** og et **Python Tool**? Giv et eksempel på hvornår du bruger hvert.

**Spørgsmål D:** Du har et RAG-flow og vil automatisk teste om det hallucinerer. Hvilken evalueringsmetrik bruger du, og hvilken flow-type implementerer evalueringen?

**Svar:**

A. 

B. 

C. 

D. 

---
## 15. Samlet Eksamensquiz: Dag 8 + 9
---

Dette er den vigtigste sektion. Svar på alle spørgsmål — de efterligner DP-100 eksamensformat.

---

**Spørgsmål 1 (Hub/Project):**
Din organisation har fem produktteams, der alle skal bruge en fælles Azure OpenAI-forbindelse og et delt GPU-compute-cluster. Hvad er den bedste måde at konfigurere dette i Azure AI Foundry?

A) Opret fem separate Hubs, én per team, og konfigurer connections i hvert Hub
B) Opret én Hub med connections og compute, og opret ét Project per team under Hub
C) Opret én Hub og ét delt Project, som alle teams arbejder i
D) Brug Azure ML workspaces i stedet for AI Foundry, da AI Foundry ikke understøtter delte ressourcer

---

**Spørgsmål 2 (Model Catalog deployment):**
Du vil deploye Phi-4-modellen fra Model Catalog til produktion. Du ønsker minimal infrastructure-opsætning og vil betale baseret på faktisk brug (tokens). Hvilken deployment-metode vælger du?

A) Managed Compute deployment med Standard_D2s_v3
B) Azure Kubernetes Service (AKS) deployment
C) Serverless API (Models as a Service)
D) Azure Container Instances

---

**Spørgsmål 3 (Flow-typer):**
Du skal bygge et flow til at evaluere om et andet flows svar er faktisk korrekte sammenlignet med kendte svar. Hvilken flow-type bruger du?

A) Standard Flow med en LLM Tool der scorer svaret
B) Chat Flow med chat_history aktiveret
C) Evaluation Flow
D) Python Flow med custom metrik-beregning

---

**Spørgsmål 4 (Connections):**
Hvor gemmes selve API-nøglerne for Prompt Flow connections?

A) I flow.dag.yaml-filen i klar tekst
B) I Azure Key Vault tilknyttet Hub'en
C) I Azure ML workspace's environment variables
D) I en .env-fil i flow-mappen

---

**Spørgsmål 5 (Tools):**
Du vil formatere et LLM's råsvar som JSON og udtrække specifikke felter, før outputtet sendes videre til næste step i dit flow. Hvilket tool bruger du?

A) LLM Tool med `response_format: json`
B) Prompt Tool med JSON-template
C) Python Tool med `@tool`-dekorator
D) En Azure Function Connection

---

**Spørgsmål 6 (Foundry vs. Azure ML):**
Hvilken af følgende påstande er KORREKT om forholdet mellem Azure AI Foundry og Azure ML?

A) Azure AI Foundry erstatter Azure ML — du kan ikke bruge Azure ML SDK med Foundry-ressourcer
B) Azure AI Foundry Hub opretter automatisk en Azure ML workspace i baggrunden
C) Azure AI Foundry er udelukkende til generative AI — klassisk ML-træning skal ske i Azure ML
D) Azure ML workspaces og AI Foundry Projects er teknisk ens og kan bruges i flæng

---

**Spørgsmål 7 (Groundedness):**
Du har bygget en RAG-chatbot og vil sikre, at den ikke hallucinerer. Hvilken built-in AI Foundry evalueringsmetrik er mest relevant?

A) Fluency — måler om sproget er grammatisk korrekt
B) Similarity — sammenligner med ground truth
C) Groundedness — måler om svaret er baseret på den retrievede kontekst
D) Coherence — måler om svaret er sammenhængende

**Svar på Eksamensquiz:**

1. 

B, én hub og fem projekter.

2. 

Serverless API deployment.

3. 



4. 



5. 

A, så jeg sikrer et struktureret output fra min LLM.
Dette er herefter nemt at behandle.

6. 

A: Forkert

B: Korrekt

C: Er overvejende enig med denne påstand, men måske det er formuleret lidt for strengt. Men modeller såsom logistisk regression og xgboost hører til i Azure ML.

D: Forstår ikke spørgsmålet. 

7. 

Groundedness. Sammenlign output med inputkontekst.

## Nøglepunkter til eksamen

**Azure AI Foundry — Struktur:**
- `Hub` = øverste niveau, delte ressourcer (connections, compute, storage), svarer til Azure ML Workspace (kind=hub)
- `Project` = isoleret arbejdsrum per team, arver ressourcer fra Hub, svarer til Azure ML Workspace (kind=project)
- Én Hub kan have mange Projects — Projects arver connections og compute fra Hub
- Oprettelse af Hub opretter automatisk en Azure ML workspace i subscription

**Azure AI Foundry — Model Catalog:**
- Model Catalog indeholder Microsoft-, OpenAI- og open-source modeller (Phi, Llama, Mistral, GPT-4o, m.fl.)
- **Serverless API**: pay-per-token, ingen compute-konfiguration, bedst til lav-volumen og cost-sensitive
- **Managed Compute**: du styrer VM-størrelse, bedst til high-volume, compliance og customization
- **Azure OpenAI Service**: dedikeret endpoint for GPT-modeller med enterprise SLA

**Prompt Flow — Flow-typer:**
- `Standard Flow`: én input → processing → ét output. Bruges til enkelt-turn tasks
- `Chat Flow`: tilføjer `chat_history` input (liste af `{role, content}`). Bruges til multi-turn samtaler
- `Evaluation Flow`: modtager output fra et andet flow + ground truth. Returnerer kvalitetsmetriks

**Prompt Flow — Tools:**
- `LLM Tool`: kalder en language model via en connection. Kræver connection-navn og deployment-navn
- `Python Tool`: vilkårlig Python-kode med `@tool`-dekorator. Bruges til data-manipulation
- `Prompt Tool`: Jinja2 template til prompt-formatering. Output sendes til LLM Tool

**Prompt Flow — Connections:**
- Connections gemmer credentials til external services (Azure OpenAI, Azure AI Search, m.fl.)
- Selve nøglerne gemmes i **Key Vault** — flows refererer til connection-navn
- Connections defineres på Hub-niveau og deles på tværs af Projects

**Prompt Flow — Deployment:**
- Deploy flow som **Online Endpoint** — samme infrastruktur som `ManagedOnlineDeployment` (dag 6+7)
- Azure ML genererer automatisk scoring-script fra flow-definitionen
- REST API input/output matcher flow-inputs og -outputs

**Evalueringsmetriks (vigtigst til RAG):**
- `Groundedness`: er svaret baseret på konteksten? (vigtigst mod hallucination)
- `Relevance`: er svaret relevant for spørgsmålet?
- `Similarity` + `F1`: kræver ground truth — måler overlap med kendte svar

**Huske-regler:**
- Hub = deling, Project = isolation
- Serverless API = pay-per-token, Managed Compute = pay-per-hour
- Chat Flow = `chat_history`, Evaluation Flow = bedømmer andres flow
- Connections + Key Vault = ingen hardkodede nøgler i flow-kode